# ML Tutor — Week 1: Python for data — in one week
**Track:** Foundation · **Lab:** Lab 1 — Out-of-range lab flagger

**Objectives this week:**
1. Consolidate core Python (variables, conditionals, loops, functions) on health data. *(CO1)*
2. Load, filter, and group clinical tables with pandas; visualize with matplotlib. *(CO1, CO4)*
3. Run the six-check data-quality sweep and build the out-of-range lab flagger. *(CO1, CO4)*

> Anti-shortcut rule: hints live in ML Tutor, revealed one tier at a time. Work here, check hints there. You are graded on unaided competence.


## Before you start (~20 min)
**Goal for this session:** Arrive able to open a Colab/Jupyter notebook, run a cell, and say what a variable is. That is all — the heavy lifting happens in studio. Reference delta: Géron-style show-then-do flow; keep deeper syntax in optional Advanced cells.

**Warm up — answer these from memory before scrolling on** (retrieval beats re-reading):
1. Name three repetitive data tasks in your pharmacy work that follow an explicit rule (if X then Y).
2. A potassium result comes back as 6.1 mEq/L. What is the normal range, and what happens next in your workflow?
3. Have you ever caught a spreadsheet error that a computer "happily" computed? What made it wrong?


In [ ]:
# SETUP — run me first (Shift+Enter)
import numpy as np
import pandas as pd

# Dataset: Inline list of 12 synthetic patient lab dicts: {"patient_id", "analyte", "value", "unit"} covering potassium, sodium, creatinine, glucose (no PHI; provided in the starter notebook).
# PLACEHOLDER: the dataset for this week arrives with the course repo under data/week-01/ .
# If a lab step expects a file, download it from the ml-tutor-labs repo's data/week-01/ folder first.

print("Setup OK — numpy", np.__version__, "· pandas", pd.__version__)


## Worked example — Types
**Lane A · the general idea:** Python values have types (int, float, str, bool). Operations are type-dependent: "3" + "4" is "34", 3 + 4 is 7. Type errors are the most common beginner crash.

**Lane B · the same idea in pharmacy terms:** A potassium of 6.1 must live as a float, not the string "6.1" — comparisons like value > 5.0 silently misbehave or crash on strings. Unit-bearing data (mEq/L) needs the number and the unit kept straight.

Below is a tiny, complete demo of this idea. Run it, read every line, change one thing, run it again.


In [ ]:
# WORKED EXAMPLE — why "6.1" (text) and 6.1 (number) behave differently
potassium_text = "6.1"              # how a CSV / LIS export delivers it
potassium = float(potassium_text)   # parse ONCE at the boundary

print(type(potassium_text), "vs", type(potassium))
print("Above 5.0?", potassium > 5.0)          # float comparison: works

try:
    potassium_text > 5.0                      # str vs float → TypeError
except TypeError as e:
    print("Comparing the STRING crashes:", e)


## 1. Represent one result
Create a dict for one lab result and print a readable sentence about it with an f-string.

**Checkpoint:** Checkpoint 1 verifies: result is a dict with the 4 required keys; value is a float (not a string!); the printed line contains patient id, analyte, value and unit.


In [ ]:
# One lab result arrives from the LIS. Represent it as a dict.
# TODO: fill in the four fields for: patient MRN-1042, potassium, 6.1, mEq/L
result = {
    "patient_id": ...,
    "analyte": ...,
    "value": ...,
    "unit": ...,
}

# TODO: print a readable sentence using an f-string, e.g.
# "Patient MRN-1042: potassium = 6.1 mEq/L"
print(...)


**Self-check 1:** run your cell, then compare its output against the checkpoint description above — does it satisfy *every* clause? Stuck? Graduated hints for this exact step live in **ML Tutor** (revealed one tier at a time, never the full answer). Fix, re-run, then move on.


## 2. Write is_out_of_range()
A function taking analyte and value, returning True/False against a reference-range dict.

**Checkpoint:** Checkpoint 2 runs 6 hidden unit tests: both tails of each range, a boundary value, and a normal value. All must return the correct boolean.


In [ ]:
# Reference ranges (adult, illustrative — teaching values only)
RANGES = {
    "potassium": (3.5, 5.0),   # mEq/L
    "sodium": (135, 145),      # mEq/L
    "creatinine": (0.6, 1.3),  # mg/dL
    "glucose": (70, 140),      # mg/dL (random)
}

def is_out_of_range(analyte, value):
    """Return True if value falls outside the reference range for analyte."""
    # TODO: look up the (low, high) pair and return the comparison
    ...

# Quick self-tests — make these pass BEFORE moving on:
print(is_out_of_range("potassium", 6.1))   # expect True
print(is_out_of_range("potassium", 4.2))   # expect False
print(is_out_of_range("sodium", 128))      # expect True


**Self-check 2:** run your cell, then compare its output against the checkpoint description above — does it satisfy *every* clause? Stuck? Graduated hints for this exact step live in **ML Tutor** (revealed one tier at a time, never the full answer). Fix, re-run, then move on.


## 3. Loop the panel
Loop all 12 results, collect flagged ones into a list.

**Checkpoint:** Checkpoint 3 verifies: flagged is a list of dicts; it contains exactly the 4 known-abnormal records from the seeded panel; order preserved.


In [ ]:
# panel: 12 synthetic results (no PHI). In Colab this same panel is also
# available via: from lab_data import panel  (course repo, week-01 data folder).
panel = [
    {"patient_id": "MRN-1042", "analyte": "potassium",  "value": 6.1,  "unit": "mEq/L"},
    {"patient_id": "MRN-1005", "analyte": "sodium",     "value": 128,  "unit": "mEq/L"},
    {"patient_id": "MRN-1091", "analyte": "creatinine", "value": 2.4,  "unit": "mg/dL"},
    {"patient_id": "MRN-1077", "analyte": "glucose",    "value": 58,   "unit": "mg/dL"},
    {"patient_id": "MRN-1010", "analyte": "potassium",  "value": 4.2,  "unit": "mEq/L"},
    {"patient_id": "MRN-1023", "analyte": "sodium",     "value": 140,  "unit": "mEq/L"},
    {"patient_id": "MRN-1034", "analyte": "creatinine", "value": 0.9,  "unit": "mg/dL"},
    {"patient_id": "MRN-1056", "analyte": "glucose",    "value": 95,   "unit": "mg/dL"},
    {"patient_id": "MRN-1067", "analyte": "potassium",  "value": 3.8,  "unit": "mEq/L"},
    {"patient_id": "MRN-1078", "analyte": "sodium",     "value": 137,  "unit": "mEq/L"},
    {"patient_id": "MRN-1089", "analyte": "creatinine", "value": 1.1,  "unit": "mg/dL"},
    {"patient_id": "MRN-1099", "analyte": "glucose",    "value": 110,  "unit": "mg/dL"},
]

flagged = ...   # TODO: start with the right empty structure

for result in panel:
    # TODO: use is_out_of_range(...) and collect the WHOLE record when it fires
    ...

print(len(flagged), "abnormal results")


**Self-check 3:** run your cell, then compare its output against the checkpoint description above — does it satisfy *every* clause? Stuck? Graduated hints for this exact step live in **ML Tutor** (revealed one tier at a time, never the full answer). Fix, re-run, then move on.


## 4. Report
Print a flag report: count + one line per abnormal result, then sanity-check it by hand.

**Checkpoint:** Checkpoint 4 verifies: header contains both counts; each flagged line contains id, analyte, value, unit AND the range; then asks you to confirm two rows by hand (verification habit).


In [ ]:
# Produce the morning safety sweep report.
# Target format:
#   FLAG REPORT — 4 abnormal of 12
#   MRN-1042  potassium   6.1 mEq/L  (range 3.5-5.0)
#   ...

# TODO: header line with counts
# TODO: one line per flagged record, including its reference range



**Self-check 4:** run your cell, then compare its output against the checkpoint description above — does it satisfy *every* clause? Stuck? Graduated hints for this exact step live in **ML Tutor** (revealed one tier at a time, never the full answer). Fix, re-run, then move on.


## Reflection — make it stick (5 min, write before you leave)
1. **Teach it:** explain your Step 4 code to a colleague in exactly 3 sentences — what goes in, what happens, what comes out. Write the sentences below.
2. **What would break this?** A classmate insists: *“Strings that look like numbers behave like numbers.”* Using what you just built, describe one concrete case from this week's lab where acting on that belief produces a wrong result — and how you would catch it.


In [ ]:
# Your reflection (write as comments)
# 1) Three sentences:
#
# 2) What would break this, concretely:
#



## Optional challenge — Homework: HW1 — Severity-tiered flagger
Extend your lab flagger: instead of True/False, classify each result as "normal", "abnormal", or "critical" using a second set of critical thresholds (e.g., potassium >= 6.0 or < 2.5 is critical). Print the report grouped by severity, critical first.

**Deliverable:** One .ipynb notebook, runnable top-to-bottom, with a 3-sentence reflection: where would this rule-based approach break down, and what would you want the computer to LEARN instead? (That question is Week 4.)


In [ ]:
# HW / challenge workspace — build it here

